# Decision trees and random forests

Adnan Masood · University of South Florida

A bank has more flagged transactions than investigators can inspect. This notebook compares one decision tree with a forest of trees, asking whether each puts more truly positive examples near the top of the review list. The data are synthetic teaching examples, not real transactions or evidence of fraud.

**Dataset:** The computer creates 2,000 fictional rows. Each row has 12 numeric features (input details), and about 3 in every 100 rows have the positive label. Seven features carry information about that label. A stratified split keeps roughly the same positive share in the 70% training group and 30% test group. Seed 11 makes the generated rows repeatable.

**Task:** Fit a tree limited to five levels and a forest of 80 trees, then score the same held-back test rows. Average precision summarizes how well positive cases rise through the ranked list. Precision@20 asks how many of the first 20 reviewed rows are positive. Twenty cases make a small sample, so one changed outcome can move that second measure noticeably.

**Compare:** Compare both models on exactly the same test rows. Use the test-set positive share as a reference: a random ranking would find positive cases at about that rate on average. The reference is not another fitted model, and the test set should not be reused to tune the models.

**Runtime:** This notebook uses scikit-learn. The forest evaluates trees one at a time to keep the run predictable and gives extra weight to the rare positive class while fitting. Average precision adds the precision gained at each recall step; it is not the same calculation as trapezoidal area under a precision–recall curve.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
X, y = make_classification(n_samples=2000, n_features=12,
    n_informative=7, weights=[.97, .03], random_state=11)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, stratify=y, test_size=.30, random_state=11)
print("Random-ranking baseline AP:", yte.mean())
models = {
 "tree": DecisionTreeClassifier(max_depth=5, random_state=11),
 "forest": RandomForestClassifier(n_estimators=80,
    max_depth=10, class_weight="balanced", n_jobs=1, random_state=11)
}
for name, model in models.items():
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    print(name, "Average precision:", average_precision_score(yte, p))
    top = p.argsort()[::-1][:20]
    print(name, "Precision@20:", yte[top].mean())


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Increase tree depth

Compare average precision and Precision@20 with the depth-5 tree.

Replace:
```python
max_depth=5
```
With:
```python
max_depth=10
```

### Increase forest size

Compare average precision, top-20 yield, and run time.

Replace:
```python
n_estimators=80
```
With:
```python
n_estimators=150
```

### Require 20 rows per tree leaf

Compare the single tree before and after limiting small leaves.

Replace:
```python
DecisionTreeClassifier(max_depth=5
```
With:
```python
DecisionTreeClassifier(min_samples_leaf=20, max_depth=5
```

### Remove forest class weights

Check whether average precision and Precision@20 change in the same direction.

Replace:
```python
class_weight="balanced"
```
With:
```python
class_weight=None
```



In [ ]:
# Compare the depth-5 tree with a depth-10 tree on the same test rows.
# Record average precision and Precision@20 for each tree and the forest.


## Break it

First remove the forest's class weights while keeping the data and split fixed. Then try a deeper tree as a separate change. Record whether average precision and Precision@20 move in the same direction. Since Precision@20 is based on just 20 cases, one changed label can shift it by five percentage points.


In [ ]:
# Run the forest with class_weight=None and compare its top-20 yield.
# Count how many positive cases correspond to each reported Precision@20.


## What to submit

- A table of tree and forest average precision and Precision@20, calculated from the same held-back rows.
- Results from changing one setting at a time, with the setting and fixed inputs stated.
- A recommendation for a limited review queue that explains the small top-20 sample, the synthetic data, and what evidence would still be needed.

## Optional extension

A hand-written transaction rule or an anomaly detector would answer a different question and is not included. Compare either separately on the same later cases before claiming that the tree or forest is more useful.
